In [1]:
import os
import re
import pandas as pd
import plotly.graph_objects as go
from scipy.stats import spearmanr
import plotly.io as pio
from typing import Dict, List, Optional

pio.renderers.default = "plotly_mimetype"

# --- Helper Functions (assuming they are defined in a previous cell) ---
def parse_config(file_path: str) -> Dict[str, str]:
    """Parses a 'key: value' or 'key = value' configuration file."""
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                parts = re.split(r'[:=]', line, 1)
                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except Exception:
        # Silently ignore if file not found or parsing fails
        pass
    return params

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str: return 0.0
    try:
        parts = time_str.split(':')
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    except (ValueError, IndexError):
        return 0.0

# --- Main Data Processing and Plotting Cell ---

BASE_OUTPUT_DIR = '/app/astra-sim/upc/output/comparison_run/'
TOPOLOGIES = ['Jellyfish', 'FoldedClos', 'Dragonfly']
TOPOLOGIES = ['FoldedClos128']
TOPOLOGIES = ['experiment6/FoldedClosECMP128']

# WORKLOAD_GROUP = 'T5_Small_grouped_ecmp'
WORKLOAD_GROUP = 'T5_Small_grouped_ecmp_sync'
# WORKLOAD_GROUP = 'T5_Small_grouped_128'
WORKLOAD_GROUP = 'T5_Small_grouped_128_sync'
WORKLOAD_GROUP = 'GPT40B_128'


all_results = []

for topo in TOPOLOGIES:
    topo_path = os.path.join(BASE_OUTPUT_DIR, topo, WORKLOAD_GROUP)
    if not os.path.isdir(topo_path):
        print(f"Directory not found for topology {topo}, skipping.")
        continue

    # A workload is a specific model parallelization strategy, e.g., "T5_Small_multiple_1_8_2_1_0..."
    for workload_name in os.listdir(topo_path):
        workload_path = os.path.join(topo_path, workload_name)
        if not os.path.isdir(workload_path):
            continue

        # This dictionary will hold the consolidated results for one workload
        workload_results = {'workload': workload_name, 'topology': topo}
        
        # Lists to collect multiple G2 and NS3 executions
        g2_exec_times = []
        g2_sim_times = []
        ns3_exec_times = []
        ns3_sim_times = []
        
        # Each workload folder contains multiple run directories, one for each simulator
        run_dirs = [d for d in os.listdir(workload_path) if d.startswith('run_')]

        for run_dir_name in run_dirs:
            run_path = os.path.join(workload_path, run_dir_name)

            # Determine which simulator this run directory is for
            sim_type = None
            if os.path.isdir(os.path.join(run_path, 'g2')):
                sim_type = 'g2'
            elif os.path.isdir(os.path.join(run_path, 'ns3')):
                sim_type = 'ns3'
            elif os.path.isdir(os.path.join(run_path, 'analytical_unaware')):
                sim_type = 'analytical_unaware'
            
            if not sim_type:
                continue # Skip if this run directory doesn't contain a known sim output

            sim_path = os.path.join(run_path, sim_type)
            sim_name = {'g2': 'G2', 'ns3': 'NS3', 'analytical_unaware': 'Analytical'}[sim_type]

            # 1. Extract Estimated Execution Time (from timing.csv)
            timing_file = next((os.path.join(sim_path, f) for f in os.listdir(sim_path) if 'trace_matched_timing.csv' in f), None)
            max_time_ns = None
            if timing_file:
                try:
                    df_timing = pd.read_csv(timing_file)
                    if 'callback_tick' in df_timing.columns:
                        max_time_ns = df_timing['callback_tick'].max()
                except Exception as e:
                    print(f"Error reading {timing_file}: {e}")
            
            # 2. Extract Simulation Time (from run_summary.txt)
            summary_params = parse_config(os.path.join(run_path, 'run_summary.txt'))
            sim_time_sec = parse_runtime(summary_params.get('total runtime', '0'))

            # Store results - collect multiple G2 and NS3 runs, single values for analytical
            if sim_type == 'g2':
                if max_time_ns is not None:
                    g2_exec_times.append(max_time_ns)
                if sim_time_sec is not None:
                    g2_sim_times.append(sim_time_sec)
            elif sim_type == 'ns3':
                if max_time_ns is not None:
                    ns3_exec_times.append(max_time_ns)
                if sim_time_sec is not None:
                    ns3_sim_times.append(sim_time_sec)
            else:
                workload_results[f'{sim_name}_Est_Exec_Time_ns'] = max_time_ns
                workload_results[f'{sim_name}_Sim_Time_sec'] = sim_time_sec
        
        # Calculate G2 statistics (mean, min, max)
        if g2_exec_times:
            workload_results['G2_Est_Exec_Time_ns'] = sum(g2_exec_times) / len(g2_exec_times)
            workload_results['G2_Est_Exec_Time_ns_min'] = min(g2_exec_times)
            workload_results['G2_Est_Exec_Time_ns_max'] = max(g2_exec_times)
        if g2_sim_times:
            workload_results['G2_Sim_Time_sec'] = sum(g2_sim_times) / len(g2_sim_times)
            workload_results['G2_Sim_Time_sec_min'] = min(g2_sim_times)
            workload_results['G2_Sim_Time_sec_max'] = max(g2_sim_times)
        
        # Calculate NS3 statistics (mean, min, max)
        if ns3_exec_times:
            workload_results['NS3_Est_Exec_Time_ns'] = sum(ns3_exec_times) / len(ns3_exec_times)
            workload_results['NS3_Est_Exec_Time_ns_min'] = min(ns3_exec_times)
            workload_results['NS3_Est_Exec_Time_ns_max'] = max(ns3_exec_times)
        if ns3_sim_times:
            workload_results['NS3_Sim_Time_sec'] = sum(ns3_sim_times) / len(ns3_sim_times)
            workload_results['NS3_Sim_Time_sec_min'] = min(ns3_sim_times)
            workload_results['NS3_Sim_Time_sec_max'] = max(ns3_sim_times)
        
        # Only add the workload if it has data
        if len(workload_results) > 2:
             all_results.append(workload_results)

# --- Create DataFrame, Plots, and Summary Tables ---
if all_results:
    df = pd.DataFrame(all_results)
    # Ensure all required simulator data is present for a workload before processing
    required_cols = ['G2_Est_Exec_Time_ns', 'G2_Est_Exec_Time_ns_min', 'G2_Est_Exec_Time_ns_max',
                     'NS3_Est_Exec_Time_ns', 'NS3_Est_Exec_Time_ns_min', 'NS3_Est_Exec_Time_ns_max',
                     'Analytical_Est_Exec_Time_ns', 
                     'G2_Sim_Time_sec', 'G2_Sim_Time_sec_min', 'G2_Sim_Time_sec_max',
                     'NS3_Sim_Time_sec', 'NS3_Sim_Time_sec_min', 'NS3_Sim_Time_sec_max',
                     'Analytical_Sim_Time_sec']
    df.dropna(subset=required_cols, inplace=True)
    df.sort_values(by=['topology', 'workload'], inplace=True)
    
    # Create output directory for plots
    # plot_output_dir = os.path.join(BASE_OUTPUT_DIR, "summary_plots")
    # os.makedirs(plot_output_dir, exist_ok=True)

    # Precompute error and speedup columns
    df['G2 Error (%)'] = ((df['G2_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
    df['AU Error (%)'] = ((df['Analytical_Est_Exec_Time_ns'] - df['NS3_Est_Exec_Time_ns']) / df['NS3_Est_Exec_Time_ns']) * 100
    df['G2 Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['G2_Sim_Time_sec']
    df['AU Speedup (x)'] = df['NS3_Sim_Time_sec'] / df['Analytical_Sim_Time_sec']

    # --- Header for LaTeX tables ---
    print(f"\n\n{'='*80}\n--- Generated LaTeX Tables ---\n{'='*80}")

    for topo in df['topology'].unique():
        topo_df = df[df['topology'] == topo].copy()
        if topo_df.empty:
            continue
        
        # Sort by NS3 execution time as the ground truth
        topo_df = topo_df.sort_values(by='NS3_Est_Exec_Time_ns').reset_index(drop=True)
        
        # Shorten workload names for cleaner plots
        topo_df['short_workload'] = topo_df['workload'].str.replace('Llama8B_last_', '', regex=False).str.replace('.seq_2048.batch_64', '', regex=False)\
            .str.replace('_', '-', regex=False)\
            .str.replace('.seq-2048.batch-1024', '', regex=False)

        print(f"\n{'='*80}\n--- Results for Topology: {topo} ---\n{'='*80}")

        # --- Plot 1: Absolute Estimated Execution Time Comparison ---
        fig_abs = go.Figure()
        
        # Add NS3 with error bars showing min/max range
        ns3_error_minus = topo_df['NS3_Est_Exec_Time_ns'] - topo_df['NS3_Est_Exec_Time_ns_min']
        ns3_error_plus = topo_df['NS3_Est_Exec_Time_ns_max'] - topo_df['NS3_Est_Exec_Time_ns']
        fig_abs.add_trace(go.Bar(
            x=topo_df['short_workload'], 
            y=topo_df['NS3_Est_Exec_Time_ns'], 
            name='NS3', 
            marker_color='lightgray',
            error_y=dict(
                type='data',
                symmetric=False,
                array=ns3_error_plus,
                arrayminus=ns3_error_minus,
                color='gray',
                thickness=1.5,
                width=4
            )
        ))
        
        # Add G2 with error bars showing min/max range
        g2_error_minus = topo_df['G2_Est_Exec_Time_ns'] - topo_df['G2_Est_Exec_Time_ns_min']
        g2_error_plus = topo_df['G2_Est_Exec_Time_ns_max'] - topo_df['G2_Est_Exec_Time_ns']
        fig_abs.add_trace(go.Bar(
            x=topo_df['short_workload'], 
            y=topo_df['G2_Est_Exec_Time_ns'], 
            name='G2', 
            marker_color='skyblue',
            error_y=dict(
                type='data',
                symmetric=False,
                array=g2_error_plus,
                arrayminus=g2_error_minus,
                color='blue',
                thickness=1.5,
                width=4
            )
        ))
        
        fig_abs.add_trace(go.Bar(x=topo_df['short_workload'], y=topo_df['Analytical_Est_Exec_Time_ns'], name='Analytical', marker_color='salmon'))
        
        fig_abs.update_layout(
            # title=f'Absolute Estimated Execution Time for {topo}',
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Estimated Execution Time (ns)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        # fig_abs.write_image(os.path.join(plot_output_dir, f"{topo}_absolute_time_comparison.pdf"), width=1400, height=600)
        fig_abs.show()

        # --- Plot 2: Ordering Comparison (Bump Chart) ---
        fig_bump = go.Figure()
        topo_df['NS3_rank'] = topo_df['NS3_Est_Exec_Time_ns'].rank(method='dense')
        topo_df['G2_rank'] = topo_df['G2_Est_Exec_Time_ns'].rank(method='dense')
        topo_df['AU_rank'] = topo_df['Analytical_Est_Exec_Time_ns'].rank(method='dense')

        for i, row in topo_df.iterrows():
            fig_bump.add_trace(go.Scatter(
                x=['G2', 'NS3', 'Analytical'],
                y=[row['G2_rank'], row['NS3_rank'], row['AU_rank']],
                mode='lines+markers',
                name=row['short_workload'],
                line=dict(color='black', width=1),
                showlegend=False
            ))
        
        fig_bump.update_layout(
            # title=f'Performance Ranking Comparison for {topo}',
            xaxis_title='Simulator',
            yaxis_title='Rank (1 is fastest)',
            yaxis=dict(autorange='reversed', tick0=1, dtick=1), # Rank 1 at top
            template='plotly_white',
            font=dict(size=16)
        )
        # fig_bump.write_image(os.path.join(plot_output_dir, f"{topo}_ranking_comparison.pdf"), width=800, height=600)
        fig_bump.show()

        # --- Plot 3: Normalized Estimated Execution Time Comparison ---
        fig_norm = go.Figure()
        
        # Add NS3 normalized with error bars (always 100% as baseline, but show min/max range)
        ns3_norm_mean = [100] * len(topo_df)
        ns3_norm_min = (topo_df['NS3_Est_Exec_Time_ns_min'] / topo_df['NS3_Est_Exec_Time_ns']) * 100
        ns3_norm_max = (topo_df['NS3_Est_Exec_Time_ns_max'] / topo_df['NS3_Est_Exec_Time_ns']) * 100
        ns3_norm_error_minus = 100 - ns3_norm_min
        ns3_norm_error_plus = ns3_norm_max - 100
        
        fig_norm.add_trace(go.Bar(
            x=topo_df['short_workload'], 
            y=ns3_norm_mean, 
            name='NS3', 
            marker_color='lightgray',
            error_y=dict(
                type='data',
                symmetric=False,
                array=ns3_norm_error_plus,
                arrayminus=ns3_norm_error_minus,
                color='gray',
                thickness=1.5,
                width=4
            )
        ))
        
        # Add G2 normalized with error bars
        g2_norm_mean = (topo_df['G2_Est_Exec_Time_ns'] / topo_df['NS3_Est_Exec_Time_ns']) * 100
        g2_norm_min = (topo_df['G2_Est_Exec_Time_ns_min'] / topo_df['NS3_Est_Exec_Time_ns']) * 100
        g2_norm_max = (topo_df['G2_Est_Exec_Time_ns_max'] / topo_df['NS3_Est_Exec_Time_ns']) * 100
        g2_norm_error_minus = g2_norm_mean - g2_norm_min
        g2_norm_error_plus = g2_norm_max - g2_norm_mean
        
        fig_norm.add_trace(go.Bar(
            x=topo_df['short_workload'], 
            y=g2_norm_mean, 
            name='G2', 
            marker_color='skyblue',
            error_y=dict(
                type='data',
                symmetric=False,
                array=g2_norm_error_plus,
                arrayminus=g2_norm_error_minus,
                color='blue',
                thickness=1.5,
                width=4
            )
        ))
        
        fig_norm.add_trace(go.Bar(x=topo_df['short_workload'], y=(topo_df['Analytical_Est_Exec_Time_ns'] / topo_df['NS3_Est_Exec_Time_ns']) * 100, name='Analytical', marker_color='salmon'))
        fig_norm.update_layout(
            # title=f'Normalized Estimated Time for {topo} (vs NS3)',
            xaxis_title='Workload (Parallelization Strategy)',
            yaxis_title='Relative Execution Time to NS3 (%)',
            barmode='group',
            xaxis_tickangle=-45,
            template='plotly_white',
            font=dict(size=16)
        )
        # fig_norm.write_image(os.path.join(plot_output_dir, f"{topo}_normalized_time_comparison.pdf"), width=1400, height=600)
        fig_norm.show()

        # --- Spearman's Rank Correlation ---
        g2_corr, _ = spearmanr(topo_df['NS3_Est_Exec_Time_ns'], topo_df['G2_Est_Exec_Time_ns'])
        au_corr, _ = spearmanr(topo_df['NS3_Est_Exec_Time_ns'], topo_df['Analytical_Est_Exec_Time_ns'])
        print(f"\n--- Spearman's Rank Correlation for {topo} (vs NS3 Estimated Time) ---")
        print(f"A value close to 1.0 indicates that the simulator preserves the performance ranking of workloads.")
        print(f" - G2 vs NS3: {g2_corr:.4f}")
        print(f" - Analytical vs NS3: {au_corr:.4f}")

        # --- Summary Table Generation (for display in notebook) ---
        summary_df = pd.DataFrame()
        summary_df['Workload'] = topo_df['short_workload']
        summary_df['NS3 Est. Time (ns)'] = topo_df['NS3_Est_Exec_Time_ns']
        summary_df['NS3 Min (ns)'] = topo_df['NS3_Est_Exec_Time_ns_min']
        summary_df['NS3 Max (ns)'] = topo_df['NS3_Est_Exec_Time_ns_max']
        summary_df['G2 Est. Time (ns)'] = topo_df['G2_Est_Exec_Time_ns']
        summary_df['G2 Min (ns)'] = topo_df['G2_Est_Exec_Time_ns_min']
        summary_df['G2 Max (ns)'] = topo_df['G2_Est_Exec_Time_ns_max']
        summary_df['G2 Error (%)'] = topo_df['G2 Error (%)']
        summary_df['AU Est. Time (ns)'] = topo_df['Analytical_Est_Exec_Time_ns']
        summary_df['AU Error (%)'] = topo_df['AU Error (%)']
        summary_df['NS3 Sim. Time (s)'] = topo_df['NS3_Sim_Time_sec']
        summary_df['NS3 Sim. Min (s)'] = topo_df['NS3_Sim_Time_sec_min']
        summary_df['NS3 Sim. Max (s)'] = topo_df['NS3_Sim_Time_sec_max']
        summary_df['G2 Sim. Time (s)'] = topo_df['G2_Sim_Time_sec']
        summary_df['G2 Sim. Min (s)'] = topo_df['G2_Sim_Time_sec_min']
        summary_df['G2 Sim. Max (s)'] = topo_df['G2_Sim_Time_sec_max']
        summary_df['G2 Speedup (x)'] = topo_df['G2 Speedup (x)']
        summary_df['AU Sim. Time (s)'] = topo_df['Analytical_Sim_Time_sec']
        summary_df['AU Speedup (x)'] = topo_df['AU Speedup (x)']
        print(f"\n--- Summary Table for {topo} ---")
        with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 250):
            display(summary_df.style.format({
                'NS3 Est. Time (ns)': '{:,.0f}', 'NS3 Min (ns)': '{:,.0f}', 'NS3 Max (ns)': '{:,.0f}',
                'G2 Est. Time (ns)': '{:,.0f}', 'G2 Min (ns)': '{:,.0f}', 'G2 Max (ns)': '{:,.0f}', 
                'G2 Error (%)': '{:+.2f}%', 'AU Est. Time (ns)': '{:,.0f}', 'AU Error (%)': '{:+.2f}%', 
                'NS3 Sim. Time (s)': '{:.2f}s', 'NS3 Sim. Min (s)': '{:.2f}s', 'NS3 Sim. Max (s)': '{:.2f}s',
                'G2 Sim. Time (s)': '{:.2f}s', 'G2 Sim. Min (s)': '{:.2f}s', 'G2 Sim. Max (s)': '{:.2f}s',
                'G2 Speedup (x)': '{:.2f}x', 'AU Sim. Time (s)': '{:.2f}s', 'AU Speedup (x)': '{:.2f}x'
            }))

        # --- LaTeX Table Generation (for papers) ---
        
        # Table 1: Estimated Execution Time
        est_df = pd.DataFrame()
        est_df['Workload'] = topo_df['short_workload']
        est_df['NS3 (s)'] = topo_df['NS3_Est_Exec_Time_ns'] / 1e9
        est_df['G2 (s)'] = topo_df['G2_Est_Exec_Time_ns'] / 1e9
        est_df['G2 Err. (%)'] = topo_df['G2 Error (%)']
        est_df['Analytic (s)'] = topo_df['Analytical_Est_Exec_Time_ns'] / 1e9
        est_df['Analytic Err. (%)'] = topo_df['AU Error (%)']
        
        # Add Average/MAPE row
        avg_row = pd.DataFrame([{
            'Workload': '\\textbf{Average / MAPE}',
            'NS3 (s)': est_df['NS3 (s)'].mean(),
            'G2 (s)': est_df['G2 (s)'].mean(),
            'G2 Err. (%)': est_df['G2 Err. (%)'].abs().mean(),
            'Analytic (s)': est_df['Analytic (s)'].mean(),
            'Analytic Err. (%)': est_df['Analytic Err. (%)'].abs().mean()
        }])
        est_df = pd.concat([est_df, avg_row], ignore_index=True)

        est_latex = est_df.to_latex(
            index=False,
            formatters={
                'NS3 (s)': "{:.3f}".format,
                'G2 (s)': "{:.3f}".format,
                'G2 Err. (%)': "{:+.2f}\%".format,
                'Analytic (s)': "{:.3f}".format,
                'Analytic Err. (%)': "{:+.2f}\%".format,
            },
            caption=f'Estimated Execution Times for {topo} Topology.',
            label=f'tab:est_times_{topo.lower()}',
            position='!htbp',
            column_format='lccccc', # l for left-aligned text, c for centered
            escape=False # To render \% correctly
        )
        # Add hline before the last row
        lines = est_latex.splitlines()
        lines.insert(-2, '\\hline')
        est_latex = '\n'.join(lines)
        est_latex = est_latex.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')


        print(f"\n--- LaTeX: Estimated Execution Time for {topo} ---")
        print(est_latex)

        # Table 2: Simulation Time
        sim_df = pd.DataFrame()
        sim_df['Workload'] = topo_df['short_workload']
        sim_df['NS3 (s)'] = topo_df['NS3_Sim_Time_sec']
        sim_df['G2 (s)'] = topo_df['G2_Sim_Time_sec']
        sim_df['G2 Speedup (x)'] = topo_df['G2 Speedup (x)']
        sim_df['Analytic (s)'] = topo_df['Analytical_Sim_Time_sec']
        sim_df['Analytic Speedup (x)'] = topo_df['AU Speedup (x)']

        # Add Average row
        avg_sim_row = pd.DataFrame([{
            'Workload': '\\textbf{Average}',
            'NS3 (s)': sim_df['NS3 (s)'].mean(),
            'G2 (s)': sim_df['G2 (s)'].mean(),
            'G2 Speedup (x)': sim_df['G2 Speedup (x)'].mean(),
            'Analytic (s)': sim_df['Analytic (s)'].mean(),
            'Analytic Speedup (x)': sim_df['Analytic Speedup (x)'].mean()
        }])
        sim_df = pd.concat([sim_df, avg_sim_row], ignore_index=True)

        sim_latex = sim_df.to_latex(
            index=False,
            formatters={
                'NS3 (s)': "{:.3f}".format,
                'G2 (s)': "{:.3f}".format,
                'G2 Speedup (x)': "{:.2f}x".format,
                'Analytic (s)': "{:.3f}".format,
                'Analytic Speedup (x)': "{:.2f}x".format,
            },
            caption=f'Simulation Times and Speedup for {topo} Topology.',
            label=f'tab:sim_times_{topo.lower()}',
            position='!htbp',
            column_format='lccccc', # l for left-aligned text, c for centered
            escape=False # To render x for speedup correctly
        )
        # Add hline before the last row
        lines = sim_latex.splitlines()
        lines.insert(-2, '\\hline')
        sim_latex = '\n'.join(lines)
        sim_latex = sim_latex.replace('\\toprule', '\\hline').replace('\\midrule', '\\hline').replace('\\bottomrule', '\\hline')
        
        print(f"\n--- LaTeX: Simulation Time for {topo} ---")
        print(sim_latex)


    # --- Overall Average Absolute Error Calculation ---
    avg_g2_abs_error = df['G2 Error (%)'].abs().mean()
    avg_au_abs_error = df['AU Error (%)'].abs().mean()
    print(f"\n{'='*80}\n--- Overall Average Absolute Error (across all topologies) ---\n{'='*80}")
    print(f"Average G2 Absolute Error vs NS3: {avg_g2_abs_error:.2f}%")
    print(f"Average Analytical Absolute Error vs NS3: {avg_au_abs_error:.2f}%")

else:
    print("No results were found to process.")



--- Generated LaTeX Tables ---

--- Results for Topology: experiment6/FoldedClosECMP128 ---



--- Spearman's Rank Correlation for experiment6/FoldedClosECMP128 (vs NS3 Estimated Time) ---
A value close to 1.0 indicates that the simulator preserves the performance ranking of workloads.
 - G2 vs NS3: 1.0000
 - Analytical vs NS3: 0.0000

--- Summary Table for experiment6/FoldedClosECMP128 ---


,Workload,NS3 Est. Time (ns),NS3 Min (ns),NS3 Max (ns),G2 Est. Time (ns),G2 Min (ns),G2 Max (ns),G2 Error (%),AU Est. Time (ns),AU Error (%),NS3 Sim. Time (s),NS3 Sim. Min (s),NS3 Sim. Max (s),G2 Sim. Time (s),G2 Sim. Min (s),G2 Sim. Max (s),G2 Speedup (x),AU Sim. Time (s),AU Speedup (x)
0,GPT40B-last-8-4-1-4-0,"1,740,363,099","1,740,363,099","1,740,363,099","1,847,487,379","1,847,487,379","1,847,487,379",+6.16%,"354,536,648",-79.63%,5591.48s,5591.48s,5591.48s,16.60s,16.60s,16.60s,336.77x,2.38s,2349.81x
1,GPT40B-last-4-8-1-4-0,"2,041,241,618","2,041,241,618","2,041,241,618","2,045,925,560","2,027,012,681","2,064,838,439",+0.23%,"194,714,270",-90.46%,3787.25s,97.39s,7477.10s,26.80s,25.35s,28.25s,141.32x,2.34s,1615.61x
2,GPT40B-last-2-8-2-4-0,"2,259,151,802","2,259,151,802","2,259,151,802","2,276,110,546","2,276,110,546","2,276,110,546",+0.75%,"291,737,115",-87.09%,7164.20s,7164.20s,7164.20s,31.25s,31.25s,31.25s,229.27x,2.50s,2861.64x
3,GPT40B-last-16-8-1-1-0,"2,564,406,139","2,564,406,139","2,564,406,139","2,624,573,611","2,624,573,611","2,624,573,611",+2.35%,"298,542,263",-88.36%,19426.54s,19426.54s,19426.54s,432.05s,432.05s,432.05s,44.96x,5.48s,3544.63x
4,GPT40B-last-8-4-2-2-0,"2,970,651,857","2,970,651,857","2,970,651,857","2,990,093,170","2,990,093,170","2,990,093,170",+0.65%,"329,667,709",-88.90%,13692.99s,13692.99s,13692.99s,67.65s,67.65s,67.65s,202.39x,3.97s,3450.55x



--- LaTeX: Estimated Execution Time for experiment6/FoldedClosECMP128 ---
\begin{table}[!htbp]
\caption{Estimated Execution Times for experiment6/FoldedClosECMP128 Topology.}
\label{tab:est_times_experiment6/foldedclosecmp128}
\begin{tabular}{lccccc}
\hline
Workload & NS3 (s) & G2 (s) & G2 Err. (%) & Analytic (s) & Analytic Err. (%) \\
\hline
GPT40B-last-8-4-1-4-0 & 1.740 & 1.847 & +6.16\% & 0.355 & -79.63\% \\
GPT40B-last-4-8-1-4-0 & 2.041 & 2.046 & +0.23\% & 0.195 & -90.46\% \\
GPT40B-last-2-8-2-4-0 & 2.259 & 2.276 & +0.75\% & 0.292 & -87.09\% \\
GPT40B-last-16-8-1-1-0 & 2.564 & 2.625 & +2.35\% & 0.299 & -88.36\% \\
GPT40B-last-8-4-2-2-0 & 2.971 & 2.990 & +0.65\% & 0.330 & -88.90\% \\
\textbf{Average / MAPE} & 2.315 & 2.357 & +2.03\% & 0.294 & +86.89\% \\
\hline
\hline
\end{tabular}
\end{table}

--- LaTeX: Simulation Time for experiment6/FoldedClosECMP128 ---
\begin{table}[!htbp]
\caption{Simulation Times and Speedup for experiment6/FoldedClosECMP128 Topology.}
\label{tab:sim_times_